# Spatial Accessibility with Pysal Access

**Authors:** Alexander Michels

In [1]:
import access
import geopandas as gpd
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
import sys

In [2]:
pop = gpd.read_file("../data/svi/SVI2018_US_tract.shp")
pop = pop[pop["STCNTY"] == "17031"]
pop["FIPS"] = pop["FIPS"].astype(str)
pop = pop.to_crs("EPSG:4326")
print(len(pop))
pop.head()

1318


,ST,STATE,ST_ABBR,STCNTY,COUNTY,FIPS,LOCATION,AREA_SQMI,E_TOTPOP,M_TOTPOP,...,F_THEME4,F_TOTAL,E_UNINSUR,M_UNINSUR,EP_UNINSUR,MP_UNINSUR,E_DAYPOP,Shape_STAr,Shape_STLe,geometry
184,17,ILLINOIS,IL,17031,Cook,17031381700,"Census Tract 3817, Cook County, Illinois",0.075986,0,11,...,-999,-999,0,11,-999.0,-999.0,551,0.000021,0.020359,"POLYGON ((-87.62892 41.80919, -87.62608 41.809..."
185,17,ILLINOIS,IL,17031,Cook,17031980000,"Census Tract 9800, Cook County, Illinois",7.679554,0,11,...,-999,-999,0,11,-999.0,-999.0,38202,0.002171,0.222431,"POLYGON ((-87.94025 42.00717, -87.93895 42.006..."
186,17,ILLINOIS,IL,17031,Cook,17031980100,"Census Tract 9801, Cook County, Illinois",1.151272,0,11,...,-999,-999,0,11,-999.0,-999.0,6449,0.000323,0.076961,"POLYGON ((-87.76230 41.79274, -87.75251 41.792..."
12180,17,ILLINOIS,IL,17031,Cook,17031020100,"Census Tract 201, Cook County, Illinois",0.158747,3836,361,...,0,0,569,245,14.8,6.3,3714,0.000044,0.027301,"POLYGON ((-87.69020 42.01682, -87.69020 42.017..."
12181,17,ILLINOIS,IL,17031,Cook,17031020200,"Census Tract 202, Cook County, Illinois",0.479524,6739,510,...,0,1,628,396,9.3,6.0,3249,0.000135,0.051852,"POLYGON ((-87.70875 42.01550, -87.70866 42.019..."


In [3]:
# load the hospitals
hos = gpd.read_file("../data/hospitals/ContinentalHospitals.shp")
hos = hos[(hos["STATE"] == "IL") & (hos["COUNTY"] == "COOK")]
hos = hos[hos["BEDS"] >= 0]
hos = hos.to_crs("EPSG:4326")
print(len(hos), sum(hos["BEDS"]))
hos.head()

66 18259


,OBJECTID,ID,NAME,ADDRESS,CITY,STATE,ZIP,ZIP4,TELEPHONE,TYPE,...,WEBSITE,STATE_ID,ALT_NAME,ST_FIPS,OWNER,TTL_STAFF,BEDS,TRAUMA,HELIPAD,geometry
507,1513,0003460644,UHS HARTGROVE HOSPITAL,5730 WEST ROOSEVELT ROAD,CHICAGO,IL,60644,NOT AVAILABLE,(773) 413-1700,PSYCHIATRIC,...,http://www.hartgrovehospital.com,0005454,NOT AVAILABLE,17,PROPRIETARY,-999,128,NOT AVAILABLE,N,POINT (-87.76746 41.86633)
508,1514,0001860612,JESSE BROWN VA MEDICAL CENTER - VA CHICAGO HEA...,820 S DAMEN STREET,CHICAGO,IL,60612,NOT AVAILABLE,(312) 569-8387,MILITARY,...,http://www.chicago.va.gov/,14003F,NOT AVAILABLE,17,GOVERNMENT - FEDERAL,-999,240,NOT AVAILABLE,N,POINT (-87.67704 41.87074)
509,1515,0003160463,PALOS COMMUNITY HOSPITAL,12251 SOUTH 80TH AVENUE,PALOS HEIGHTS,IL,60464,NOT AVAILABLE,(708) 923-4000,GENERAL ACUTE CARE,...,http://www.paloscommunityhospital.org,140062,NOT AVAILABLE,17,NON-PROFIT,-999,377,NOT AVAILABLE,Y,POINT (-87.81344 41.66921)
510,1516,0003660628,ROSELAND COMMUNITY HOSPITAL,45 W 111TH STREET,CHICAGO,IL,60628,NOT AVAILABLE,(773) 995-3000,GENERAL ACUTE CARE,...,http://www.roselandhospital.org,140068,NOT AVAILABLE,17,NON-PROFIT,-999,115,NOT AVAILABLE,N,POINT (-87.62536 41.69223)
511,1517,0002360302,WEST SUBURBAN MEDICAL CENTER,3 ERIE COURT,OAK PARK,IL,60302,NOT AVAILABLE,(708) 383-6200,GENERAL ACUTE CARE,...,http://www.westsuburbanmc.com/Home.aspx,140049,NOT AVAILABLE,17,PROPRIETARY,-999,172,NOT AVAILABLE,N,POINT (-87.77618 41.89140)


In [4]:
access.Datasets.available_datasets()


chi_times: Cost matrix with travel times from each Chicago Census Tract to all others.

chi_doc: Doctor and dentist counts for each Chicago Census Tract.

chi_pop: Population counts for each Chicago Census Tract.

chi_doc_geom: Doctor and dentist counts for each Chicago Census Tract along with geometric representations for Census Tracts.

chi_pop_geom: Population counts for each Chicago Census Tract along with geometric representations for Census Tracts.

chi_euclidean: Euclidean distance cost matrix with distances from each demand Chicago Census Tract to all others.

chi_euclidean_neighbors: Euclidean distance cost matrix with distances from each supply Census Tract to all others.

cook_county_hospitals: Contains data for each hospital location in Cook County including X Y coordinates.

cook_county_hospitals_geom: Contains data for each hospital location in Cook County including X Y coordinates, and geometric points for each hospital.

cook_county_tracts: Geometric representation of 

In [5]:
cook_ct = access.Datasets.load_data("cook_county_tracts")
cook_ct["geoid"] = cook_ct["geoid"].astype(str)
cook_ct = cook_ct.to_crs("EPSG:4326")
cook_ct.head()

,GEO_ID,STATE,COUNTY,TRACT,NAME,LSAD,CENSUSAREA,geoid,geometry
0,1400000US17031440300,17,031,440300,4403,Tract,0.499,17031440300,"POLYGON ((-87.61467 41.74198, -87.61462 41.740..."
1,1400000US17031440600,17,031,440600,4406,Tract,0.259,17031440600,"POLYGON ((-87.61453 41.73650, -87.61450 41.735..."
2,1400000US17031440700,17,031,440700,4407,Tract,0.126,17031440700,"POLYGON ((-87.60967 41.73658, -87.60725 41.736..."
3,1400000US17031460200,17,031,460200,4602,Tract,0.232,17031460200,"POLYGON ((-87.55276 41.74464, -87.55150 41.744..."
4,1400000US17031460600,17,031,460600,4606,Tract,0.101,17031460600,"POLYGON ((-87.55629 41.73732, -87.55558 41.737..."


In [6]:
hospitals = gpd.sjoin(cook_ct[["geoid", "geometry"]], hos[["BEDS", "geometry"]], how="inner", predicate="intersects")
hospitals = hospitals.groupby("geoid").agg({"BEDS": 'sum', 'geoid': 'last', "geometry": 'last'})
print(len(hospitals), sum(hospitals["BEDS"]))
hospitals.head()

59 18259


,BEDS,geoid,geometry
geoid,,,
17031031000,189,17031031000,"POLYGON ((-87.66944 41.96888, -87.67102 41.968..."
17031031300,115,17031031300,"POLYGON ((-87.65215 41.97644, -87.64844 41.977..."
17031031400,184,17031031400,"POLYGON ((-87.64257 41.96245, -87.64225 41.961..."
17031032100,134,17031032100,"POLYGON ((-87.65634 41.96176, -87.65514 41.961..."
17031040300,279,17031040300,"POLYGON ((-87.70530 41.97764, -87.70576 41.979..."


In [7]:
hospitals["geoid"].is_unique

True

In [8]:
# get car travel times
car = pd.read_csv("../data/travel_times/17031-output-TRACT-CAR.csv")
car["origin"] = car["origin"].astype(str)
car["destination"] = car["destination"].astype(str)
car.head()

,origin,destination,minutes
0,17031010100,17007010100,101.35
1,17031010100,17007010200,96.10
2,17031010100,17007010300,96.43
3,17031010100,17007010400,96.17
4,17031010100,17007010500,107.13


In [9]:
# calculate the gravity-based access
A = access.Access(
    demand_df=pop,
    demand_index="FIPS",
    demand_value="E_TOTPOP",
    supply_df=hospitals,
    supply_index="geoid",
    supply_value="BEDS",
    cost_df=car,
    cost_origin="origin",
    cost_dest="destination",
    cost_name="minutes",
    neighbor_cost_df=car,
    neighbor_cost_origin="origin",
    neighbor_cost_dest="destination",
    neighbor_cost_name="minutes"
)

In [10]:
gravity = access.weights.gravity(scale=60, alpha=-1, min_dist=1)
fn30 = access.weights.step_fn({10: 1, 20: 0.68, 30: 0.22})
fn60 = access.weights.step_fn({20: 1, 40: 0.68, 60: 0.22})
gaussian = access.weights.gaussian(20)

In [11]:
A.weighted_catchment(name="gravity", weight_fn=gravity)
A.fca_ratio(name="fca", max_cost=15)
A.fca_ratio(name="fca", max_cost=30) 
A.fca_ratio(name="fca60", max_cost=60)
A.fca_ratio(name="fca90", max_cost=90)
A.two_stage_fca(name="2sfca", max_cost=60)
A.enhanced_two_stage_fca(name="2sfca30", weight_fn=fn30)
A.enhanced_two_stage_fca(name="2sfca60", weight_fn=fn60)
A.enhanced_two_stage_fca(name="g2sfca", weight_fn=gaussian)
A.three_stage_fca(name="3sfca")
A.raam(name="raam", tau=60)
A.raam(name="raam30", tau=30)

access INFO     :: Overwriting fca_BEDS.


,raam30_BEDS
FIPS,
17031010100,1.353203
17031010201,1.278645
17031010202,1.375311
17031010300,1.428868
17031010400,1.375371
...,...
17031843700,1.265745
17031843800,1.431567
17031843900,1.344395


In [12]:
print(len(A.access_df))
A.access_df.head()

1318


,E_TOTPOP,gravity_BEDS,fca_BEDS,fca60_BEDS,fca90_BEDS,2sfca_BEDS,2sfca30_BEDS,2sfca60_BEDS,g2sfca_BEDS,3sfca_BEDS,raam_BEDS,raam30_BEDS
FIPS,,,,,,,,,,,,
17031010100,4522,28947.351633,0.005191,0.003693,0.003495,0.002790,0.003988,0.002484,0.002699,0.005337,1.151754,1.353203
17031010201,7039,30688.468786,0.004612,0.003705,0.003495,0.003022,0.004222,0.002714,0.002867,0.005148,1.115558,1.278645
17031010202,2852,29613.408700,0.004703,0.003669,0.003495,0.002956,0.004354,0.002645,0.002820,0.005539,1.173011,1.375311
17031010300,6650,29395.681764,0.004847,0.003738,0.003495,0.002931,0.004445,0.002648,0.002814,0.005940,1.187285,1.428868
17031010400,5153,31556.255500,0.004244,0.003601,0.003495,0.003050,0.002963,0.003023,0.003120,0.004095,1.126623,1.375371


In [13]:
access_result = A.access_df.merge(cook_ct[["geoid", "geometry"]], left_index=True, right_on="geoid").set_index("geoid")
access_result = gpd.GeoDataFrame(access_result)
access_result = access_result.drop(columns=["E_TOTPOP"])
print(len(access_result))
print(type(access_result))
access_result.head()

1318
<class 'geopandas.geodataframe.GeoDataFrame'>


,gravity_BEDS,fca_BEDS,fca60_BEDS,fca90_BEDS,2sfca_BEDS,2sfca30_BEDS,2sfca60_BEDS,g2sfca_BEDS,3sfca_BEDS,raam_BEDS,raam30_BEDS,geometry
geoid,,,,,,,,,,,,
17031010100,28947.351633,0.005191,0.003693,0.003495,0.002790,0.003988,0.002484,0.002699,0.005337,1.151754,1.353203,"POLYGON ((-87.66507 42.01926, -87.66604 42.019..."
17031010201,30688.468786,0.004612,0.003705,0.003495,0.003022,0.004222,0.002714,0.002867,0.005148,1.115558,1.278645,"POLYGON ((-87.68281 42.01947, -87.68163 42.019..."
17031010202,29613.408700,0.004703,0.003669,0.003495,0.002956,0.004354,0.002645,0.002820,0.005539,1.173011,1.375311,"POLYGON ((-87.67464 42.01263, -87.67547 42.012..."
17031010300,29395.681764,0.004847,0.003738,0.003495,0.002931,0.004445,0.002648,0.002814,0.005940,1.187285,1.428868,"POLYGON ((-87.66383 42.01612, -87.66340 42.015..."
17031010400,31556.255500,0.004244,0.003601,0.003495,0.003050,0.002963,0.003023,0.003120,0.004095,1.126623,1.375371,"POLYGON ((-87.66156 42.01041, -87.66108 42.009..."


In [14]:
corr = access_result.corr(method='kendall')
corr.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1)

,gravity_BEDS,fca_BEDS,fca60_BEDS,fca90_BEDS,2sfca_BEDS,2sfca30_BEDS,2sfca60_BEDS,g2sfca_BEDS,3sfca_BEDS,raam_BEDS,raam30_BEDS
gravity_BEDS,1.000000,0.418928,-0.451956,-0.142672,0.678072,0.648677,0.740050,0.797307,0.481011,-0.663228,-0.656957
fca_BEDS,0.418928,1.000000,-0.240970,-0.070710,0.403864,0.361909,0.325709,0.375433,0.387962,-0.422593,-0.403532
fca60_BEDS,-0.451956,-0.240970,1.000000,0.094149,-0.391995,-0.365058,-0.476591,-0.487398,-0.227840,0.325052,0.318597
fca90_BEDS,-0.142672,-0.070710,0.094149,1.000000,-0.152429,-0.096347,-0.143052,-0.138186,-0.045983,0.096019,0.082770
2sfca_BEDS,0.678072,0.403864,-0.391995,-0.152429,1.000000,0.539095,0.791292,0.748613,0.341379,-0.468534,-0.450638
2sfca30_BEDS,0.648677,0.361909,-0.365058,-0.096347,0.539095,1.000000,0.608575,0.695556,0.645446,-0.589144,-0.594076
2sfca60_BEDS,0.740050,0.325709,-0.476591,-0.143052,0.791292,0.608575,1.000000,0.869841,0.333898,-0.464624,-0.449428
g2sfca_BEDS,0.797307,0.375433,-0.487398,-0.138186,0.748613,0.695556,0.869841,1.000000,0.404641,-0.527022,-0.514542
3sfca_BEDS,0.481011,0.387962,-0.227840,-0.045983,0.341379,0.645446,0.333898,0.404641,1.000000,-0.659424,-0.666412
raam_BEDS,-0.663228,-0.422593,0.325052,0.096019,-0.468534,-0.589144,-0.464624,-0.527022,-0.659424,1.000000,0.898447


In [15]:
access_result.to_file("../data/accessibility/CookHospitals.gpkg", driver="GPKG")